In [ ]:
import kagglehub

d:\MSc\3. Spring 2026\CSE715\Project\vae-audio-clustering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Download latest version
path = kagglehub.dataset_download("thisisjibon/banglabeats3sec")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
import shutil, random, csv

In [ ]:
SOURCE_DIR = Path("C:/Users/User/Downloads/VAE project/archive/wavs3sec") # -> DO NOT REMOVE
# SOURCE_DIR = Path(input("Enter the path where the above Kaggle dataset was downloaded in your device: "))
DEST_DIR = Path("../../../data/audio/bn_clips/")
METADATA_FILE = Path("../../../data/metadata/metadata_bn.csv")

In [ ]:
def find_audio_files(subdir):
    return [f for f in subdir.iterdir() if f.is_file() and f.suffix.lower() == ".wav"]

In [ ]:
def transfer_audio_files(files, subdir, metadata_rows, copy_instead_of_move=True):
    for src_file in files:
        dst_file = DEST_DIR / src_file.name
        if dst_file.exists():
            stem = dst_file.stem
            suffix = dst_file.suffix
            counter = 1
            while True:
                new_name = f"{stem}_{counter}{suffix}"
                candidate = DEST_DIR / new_name
                if not candidate.exists():
                    dst_file = candidate
                    break
                counter += 1
        shutil.copy2(src_file, dst_file) if copy_instead_of_move else shutil.move(str(src_file), str(dst_file))
        metadata_rows.append({
            "label": dst_file.name,
            "genre": subdir.name
        })
        
    return metadata_rows

In [ ]:
random.seed(42)

In [ ]:
DEST_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
metadata_rows = []

In [ ]:
for subdir in SOURCE_DIR.iterdir():
    try:
        if not subdir.is_dir(): continue
        files = find_audio_files(subdir=subdir)
        # n = len(files)
        # if n == 0: continue
        # t = floor((k / 100) * n)
        # if k > 0 and t == 0: t = 1
        # t = min(t, n)
        # print(t)
        # total += t
        # chosen_files = random.sample(files, t)
        metadata_rows = transfer_audio_files(files=files, subdir=subdir, metadata_rows=metadata_rows)
    except Exception as e:
        print(e)

In [ ]:
with open(METADATA_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["label", "genre"])
    writer.writeheader()
    writer.writerows(metadata_rows)